In [4]:
# =====================================
# 1. Import Libraries
# =====================================
import numpy as np
import pandas as pd
import string

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.utils import to_categorical

In [5]:
# =====================================
# 2. Load Dataset
# =====================================
df = pd.read_csv("qoute_dataset.csv")

quotes = df['quote']

In [6]:
# =====================================
# 3. Text Cleaning
# =====================================
quotes = quotes.str.lower()

translator = str.maketrans('', '', string.punctuation)

quotes = quotes.apply(lambda x: x.translate(translator))

In [7]:
# =====================================
# 4. Tokenization
# =====================================
vocab_size = 2000

tokenizer = Tokenizer(num_words=vocab_size)

tokenizer.fit_on_texts(quotes)

word_index = tokenizer.word_index

In [8]:
# =====================================
# 5. Convert Text → Sequences
# =====================================
sequences = tokenizer.texts_to_sequences(quotes)

X = []
y = []

for seq in sequences:

    for i in range(1, len(seq)):

        input_seq = seq[:i]
        output_word = seq[i]

        X.append(input_seq)
        y.append(output_word)

In [9]:
# =====================================
# 6. Padding
# =====================================
max_len = 20

X = pad_sequences(X, maxlen=max_len, padding='pre')

y = np.array(y)

In [10]:
# =====================================
# 7. One Hot Encoding
# =====================================
y = to_categorical(y, num_classes=vocab_size)

In [11]:
# =====================================
# 8. Build Model
# =====================================
model = Sequential()

model.add(Embedding(vocab_size, 150))

model.add(Bidirectional(LSTM(150)))

model.add(Dropout(0.2))

model.add(Dense(vocab_size, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 150)         300000    
                                                                 
 bidirectional (Bidirection  (None, 300)               361200    
 al)                                                             
                                                                 
 dropout (Dropout)           (None, 300)               0         
                                                                 
 dense (Dense)               (None, 2000)              602000    
                                                                 
Total params: 1263200 (4.82 MB)
Trainable params: 1263200 (4.82 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [12]:
# =====================================
# 9. Train Model
# =====================================
history = model.fit(
    X,
    y,
    epochs=60,
    batch_size=128,
    validation_split=0.1
)

Epoch 1/60


536/536 [==============================] - 50s 85ms/step - loss: 5.9211 - accuracy: 0.0484 - val_loss: 5.7366 - val_accuracy: 0.0712
Epoch 2/60
536/536 [==============================] - 53s 100ms/step - loss: 5.4628 - accuracy: 0.0969 - val_loss: 5.3831 - val_accuracy: 0.1123
Epoch 3/60
536/536 [==============================] - 45s 83ms/step - loss: 5.1809 - accuracy: 0.1202 - val_loss: 5.2650 - val_accuracy: 0.1264
Epoch 4/60
536/536 [==============================] - 46s 86ms/step - loss: 4.9862 - accuracy: 0.1344 - val_loss: 5.1959 - val_accuracy: 0.1292
Epoch 5/60
536/536 [==============================] - 53s 99ms/step - loss: 4.8182 - accuracy: 0.1460 - val_loss: 5.1701 - val_accuracy: 0.1317
Epoch 6/60
536/536 [==============================] - 51s 95ms/step - loss: 4.6675 - accuracy: 0.1552 - val_loss: 5.1645 - val_accuracy: 0.1314
Epoch 7/60
536/536 [==============================] - 51s 96ms/step - loss: 4.5187 - accuracy: 0.1658 - val_loss: 5.1892 - val_accura

In [13]:
# =====================================
# 10. Save Model
# =====================================
model.save("next_word_model.h5")

c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [14]:
# =====================================
# 11. Index → Word Mapping
# =====================================
index_to_word = {}

for word, index in word_index.items():

    index_to_word[index] = word

In [15]:
# =====================================
# 12. Predict Next Word (Top-K Sampling)
# =====================================
def predict_next_word(text):

    text = text.lower()

    seq = tokenizer.texts_to_sequences([text])[0]

    seq = pad_sequences([seq], maxlen=max_len, padding='pre')

    preds = model.predict(seq, verbose=0)[0]

    top_indices = preds.argsort()[-5:][::-1]

    next_index = np.random.choice(top_indices)

    return index_to_word.get(next_index, "")

In [16]:
# =====================================
# 13. Generate Sentence
# =====================================
def generate_text(seed_text, n_words):

    for i in range(n_words):

        next_word = predict_next_word(seed_text)

        if next_word == "":
            break

        seed_text += " " + next_word

    return seed_text

In [17]:
# =====================================
# 14. Test Prediction
# =====================================
seed = "are you a"

result = generate_text(seed, 10)

print(result)

are you a never get of doing the right story like telling the


In [18]:
seed = "are you a"

next_word = predict_next_word(seed)

print(next_word)

that


In [19]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [20]:
with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)